# 🔴 Interagindo com Redis usando Python

Este notebook é um guia didático e prático que demonstra como interagir com o **Redis** (banco de dados NoSQL do tipo **Chave-Valor**) utilizando a linguagem Python.

## 🛠️ O que é o Redis?
O Redis (Remote Dictionary Server) é um armazenamento de estrutura de dados em memória, usado como banco de dados, cache e message broker. Ele é extremamente rápido porque mantém os dados na **memória RAM**, persistindo-os em disco de forma assíncrona.

### Resumo Conceitual

| Propriedade | Detalhes |
|---|---|
| **Paradigma** | Chave-Valor (Key-Value Store) |
| **Linguagem de Consulta** | Comandos Redis (SET, GET, DEL, HSET, LPUSH, etc.) |
| **Armazenamento** | Dados em memória RAM com persistência opcional em disco (RDB/AOF) |
| **Quando usar** | Cache de alta performance, sessões de usuário, filas de mensagens, contadores em tempo real, leaderboards, pub/sub |
| **Quando NÃO usar** | Dados relacionais complexos, consultas com JOINs, dados que excedem a memória RAM disponível, persistência como requisito crítico |

### Detalhes da Conexão Local (Docker Compose):
- **Host:** `localhost`
- **Porta:** `6379`
- **Autenticação:** Nenhuma (configuração padrão local)

## 📋 Pré-requisitos

Antes de executar este notebook, certifique-se de que:

1. O **Docker** está instalado e em execução na sua máquina.
2. Os containers do projeto foram iniciados com `make up` ou `docker compose up -d`.
3. O container `redis` está rodando (verifique com `docker compose ps`).

> **💡 Dica:** O Redis inicia quase instantaneamente, diferente de outros bancos como o Cassandra que podem demorar 1-2 minutos.

## 2. Conectando ao Banco de Dados
Vamos importar a biblioteca e criar uma instância de conexão. 

> **💡 Conceito-Chave:** O parâmetro `decode_responses=True` é fundamental! Por padrão, o Redis retorna dados como **bytes** (`b'texto'`). Ao ativar esse parâmetro, as respostas são automaticamente convertidas para **strings** normais do Python, facilitando a manipulação.

**Saída esperada:**
```
✅ Conexão com Redis estabelecida com sucesso!
```

In [ ]:
import redis

# Criar conexão com o cliente local Redis
# - host: endereço do servidor (localhost porque o container Docker mapeia a porta)
# - port: porta padrão do Redis
# - decode_responses: converte bytes para strings automaticamente
try:
    client = redis.Redis(host='localhost', port=6379, decode_responses=True)
    
    # O comando ping() testa se a conexão está funcionando
    # Retorna True se o servidor respondeu com PONG
    if client.ping():
        print("✅ Conexão com Redis estabelecida com sucesso!")
except Exception as e:
    print(f"❌ Erro ao conectar ao Redis: {e}")
    print("Certifique-se de que o container do Redis está rodando (use 'make up' ou 'docker compose up -d')")

---
## 3. Operações CRUD Básicas (Chaves do Tipo String)
O tipo mais básico de valor no Redis é a **String** (que pode conter texto, números ou até binários serializados).

> **💡 Conceito-Chave:** No Redis, a convenção de nomenclatura de chaves usa `:` (dois pontos) como separador lógico para criar uma hierarquia visual. Exemplo: `usuario:1:nome` indica que é o campo `nome` do `usuario` com ID `1`. Isso é apenas uma convenção — o Redis trata como uma string única.

### 3.1 CREATE — Inserir dados
O comando `SET` armazena um valor associado a uma chave.

Sintaxe: 

```python
client.set(<chave>,<valor>)
```

**Saída esperada:**
```
✍️ Chave 'usuario:1:nome' criada com o valor: 'Carlos Silva'
```

In [ ]:
# SET define o valor de uma chave
# Se a chave não existir, ela será criada automaticamente
client.set('usuario:1:nome', 'Carlos Silva')
print("Dados inserido")

### 3.2 READ — Ler dados
O comando `GET` recupera o valor armazenado em uma chave. Retorna `None` se a chave não existir.

Sintaxe:

```python
client.get(<chave>) # -> retorna o valor
```

**Saída esperada:**
```
📖 Valor de 'usuario:1:nome': Carlos Silva
📖 Valor de 'chave:inexistente': None
```

In [ ]:
# GET recupera o valor associado à chave
nome_usuario = client.get('usuario:1:nome')
print(f"📖 Valor de 'usuario:1:nome': {nome_usuario}")

# Se tentarmos ler uma chave que não existe, o retorno será None
valor_inexistente = client.get('chave:inexistente')
print(f"📖 Valor de 'chave:inexistente': {valor_inexistente}")

### 3.3 UPDATE — Atualizar dados
No Redis, **não existe um comando UPDATE separado**. Um novo `SET` na mesma chave simplesmente sobrescreve o valor antigo. Essa simplicidade é uma das características do modelo chave-valor.

**Saída esperada:**
```
🔄 Valor ANTES da atualização: Carlos Silva
🔄 Valor APÓS a atualização: Carlos Souza
```

In [ ]:
# Lembrar: não existe "UPDATE" no Redis — o SET sobrescreve
print(f"🔄 Valor ANTES da atualização: {client.get('usuario:1:nome')}")

client.set('usuario:1:nome', 'Carlos Souza')

print(f"🔄 Valor APÓS a atualização: {client.get('usuario:1:nome')}")

### 3.4 DELETE — Deletar dados
O comando `DEL` remove uma chave e seu valor do banco. O comando `EXISTS` pode ser utilizado para verificar se a chave existe (retorna `1` se existir, `0` caso contrário).

Sintaxe:

```python
client.delete(<chave>)
client.exists(<chave>)
```

**Saída esperada:**
```
🗑️ Chave deletada com sucesso.
❓ A chave 'usuario:1:nome' ainda existe? Não
```

In [ ]:
# DEL remove a chave do banco de dados
client.delete('usuario:1:nome')
print("🗑️ Chave deletada com sucesso.")

# EXISTS verifica se a chave ainda está presente
# Retorna 1 (True) se existir, 0 (False) se não existir
existe = client.exists('usuario:1:nome')
print(f"❓ A chave 'usuario:1:nome' ainda existe? {'Sim' if existe else 'Não'}")

---
## 4. Controle de Expiração (TTL — Time to Live)
Uma das funcionalidades mais poderosas do Redis é a capacidade de definir uma **expiração automática** para as chaves. Após o tempo expirar, a chave é removida automaticamente do banco.

> **💡 Caso de Uso Real:** O TTL é amplamente utilizado para gerenciar **sessões de usuário** (ex: tokens JWT que expiram em 30 minutos), **cache temporário** (ex: resultado de uma API que muda a cada 5 minutos) e **rate limiting** (ex: limitar 100 requisições por minuto por IP).

**Sintaxe: SET com expiration -> SETEX**

```phthon
client.setex(<chave>, <tempo segundos>, <valor>) # Inserir chave-valor com tempo de expiração
```

**Sintaxe: TTL retorna tempo restante em segundos**


```phthon
client.ttl(<chave>) # Inserir chave-valor com tempo de expiração
```

In [ ]:
import time

# SETEX = SET com EXpiration
# Parâmetros: (chave, tempo_em_segundos, valor)
# A chave será automaticamente removida após o tempo definido
client.setex('sessao:token', 5, 'jwt_token_exemplo_123')
print("🔑 Token de sessão inserido com expiração de 5 segundos.")

# TTL (Time To Live) retorna o tempo restante de vida da chave em segundos
# Retorna -1 se a chave não tem expiração, -2 se a chave não existe
ttl_inicial = client.ttl('sessao:token')
print(f"⏳ TTL inicial: {ttl_inicial} segundos")

# Aguardar 30 segundos para ver o TTL diminuir
time.sleep(30)
ttl_restante = client.ttl('sessao:token')
valor_token = client.get('sessao:token')
print(f"⏳ TTL após 30 segundos: {ttl_restante} segundos (Valor: {valor_token})")

# Aguardar mais 60 segundos (totalizando 6, estourando os 60 segundos de TTL)
print("Aguardando a chave expirar...")
time.sleep(60)
valor_expirado = client.get('sessao:token')
print(f"🗑️ Valor recuperado após expiração: {valor_expirado} (Chave expirou automaticamente!)")

---
## 5. Estruturas de Dados Avançadas
O Redis vai muito além de simples strings! Ele suporta várias estruturas de dados nativas, o que o torna extremamente versátil. Vamos explorar as três mais utilizadas:

| Estrutura | Descrição | Caso de Uso Típico |
|---|---|---|
| **Hashes** | Dicionários (pares campo-valor dentro de uma chave) | Perfis de usuário, objetos estruturados |
| **Lists** | Listas ordenadas por ordem de inserção | Filas de tarefas, histórico de ações |
| **Sets** | Conjuntos de valores únicos (sem duplicatas) | Tags, seguidores, lista de presença |

### 5A. Hashes (Dicionários/Objetos)
Hashes são ótimos para representar **objetos estruturados**, contendo múltiplos campos e valores dentro de uma única chave principal. Pense neles como um "mini dicionário Python" armazenado no Redis.

> **💡 Vantagem sobre Strings:** Em vez de criar múltiplas chaves (`usuario:100:nome`, `usuario:100:email`, `usuario:100:idade`), você pode armazenar tudo em um único Hash na chave `usuario:100`. Isso é mais eficiente em memória e organização.

**Sintaxa HSET - hash set**

```python
client.hset(<chave principal>, mapping=<dicionario>)
```

In [ ]:
chave_hash = 'usuario:100'

# HSET com mapping insere múltiplos campos de uma vez no Hash
# É equivalente a rodar vários HSET individuais
client.hset(chave_hash, mapping={
    'nome': 'Alice Silva',
    'email': 'alice@email.com',
    'idade': '28',
    'cidade': 'João Pessoa'
})
print(f"📝 Hash criado em '{chave_hash}'")

**Sintaxe: HGET -> hash get**

```python
client.hget(<chave principal>, <chave do dict>)
```

In [ ]:
# HGET obtém o valor de um campo específico do Hash
nome = client.hget(chave_hash, 'nome')
print(f"👤 Nome do usuário: {nome}")





**Sintaxe: HGETALL -> hash get all**

```python
client.hgetall(<chave principal>)
```

In [ ]:
# HGETALL retorna todos os campos e valores do Hash como um dicionário Python
dados_usuario = client.hgetall(chave_hash)
print(f"📦 Objeto completo: {dados_usuario}")

**Sintaxe: HSET -> hash SET**

```python
client.hset(<chave hash>, <chave dict>, <novo valor dict>)
```

In [ ]:
# HSET em um campo existente atualiza apenas aquele campo (sem afetar os demais)
client.hset(chave_hash, 'idade', '29')
idade_atualizada = client.hget(chave_hash, 'idade')
print(f"🔄 Idade atualizada para: {idade_atualizada}")

**Sintaxe: HDSEL -> hash DEL**

```python
client.hdel(<chave hash>, <chave dict>) # remove chave-valor do dict
```

In [ ]:
# HDEL remove um campo específico do Hash (sem apagar os outros campos)
client.hdel(chave_hash, 'cidade')
dados_finais = client.hgetall(chave_hash)
print(f"🗑️ Hash após deletar o campo 'cidade': {dados_finais}")



**Sintaxe: delete**

```python
client.delete(<chave hash>) # remove todo o objeto
```

In [ ]:
# Limpar chave para manter ambiente organizado
client.delete(chave_hash)

### 5B. Listas (Fila / Pilha)
Listas do Redis são coleções de strings **ordenadas pela ordem de inserção**. 

Você pode adicionar elementos no início (`LPUSH`) ou no fim (`RPUSH`), tornando-as úteis como **filas (FIFO)** ou **pilhas (LIFO)**.

> **💡 Analogia:** Imagine uma fila de banco. Novas pessoas entram no final da fila (`RPUSH`), e o próximo a ser atendido sai do início (`LPOP`). Se alguém tem prioridade, pode "furar a fila" e entrar no início (`LPUSH`).

**Saída esperada:**
```
📋 Tamanho da lista de tarefas: 3
🔍 Lista completa (note a ordem): ['Corrigir bug crítico em produção', 'Enviar e-mail para o cliente', 'Revisar PR pendente']
✅ Tarefa concluída e removida da fila: 'Corrigir bug crítico em produção'
🔍 Lista restante: ['Enviar e-mail para o cliente', 'Revisar PR pendente']
```

In [ ]:
chave_lista = 'tarefas:urgentes'
client.delete(chave_lista)  # Limpar se já existir de execuções anteriores

# RPUSH adiciona itens no FIM da lista (Right Push)
client.rpush(chave_lista, 'Enviar e-mail para o cliente')
client.rpush(chave_lista, 'Revisar PR pendente')

# LPUSH adiciona itens no INÍCIO da lista (Left Push)
# Isso simula uma tarefa urgente "furando a fila"
client.lpush(chave_lista, 'Corrigir bug crítico em produção')

# LLEN retorna o tamanho (comprimento) da lista
tamanho = client.llen(chave_lista)
print(f"📋 Tamanho da lista de tarefas: {tamanho}")

# LRANGE retorna um subconjunto da lista (0 = primeiro, -1 = último)
# Usando 0 a -1 trazemos a lista completa
itens = client.lrange(chave_lista, 0, -1)
print(f"🔍 Lista completa (note a ordem): {itens}")


In [ ]:

# LPOP remove e retorna o PRIMEIRO item da lista
# Isso simula "atender o próximo da fila"
primeira_tarefa = client.lpop(chave_lista)
print(f"✅ Tarefa concluída e removida da fila: '{primeira_tarefa}'")

# Verificar o estado restante da lista
itens_restantes = client.lrange(chave_lista, 0, -1)
print(f"🔍 Lista restante: {itens_restantes}")

# Limpar chave
client.delete(chave_lista)

### 5C. Sets (Conjuntos Únicos e Não Ordenados)
Sets são coleções de strings **únicas** (sem duplicidade) e **não ordenadas**. Úteis para tags, listas de presença, seguidores, etc.

> **💡 Conceito-Chave:** A principal diferença entre Sets e Lists é que o Set **ignora duplicatas automaticamente**. Se você tentar adicionar um valor que já existe, ele simplesmente não será duplicado.

**Saída esperada:**
```
🏷️ Tags do Post (repare que 'nosql' aparece apenas uma vez): {'programacao', 'nosql', 'tecnologia'}
❓ Contém a tag 'python'? Não
❓ Contém a tag 'nosql'? Sim
🗑️ Tags restantes após remover 'tecnologia': {'programacao', 'nosql'}
```

In [ ]:
chave_set = 'tags:post:42'
client.delete(chave_set)  # Limpar se já existir de execuções anteriores

# SADD adiciona membros ao Set
client.sadd(chave_set, 'tecnologia')
client.sadd(chave_set, 'programacao')
client.sadd(chave_set, 'nosql')

# Tentar adicionar item duplicado — não terá efeito!
client.sadd(chave_set, 'nosql')

# SMEMBERS retorna todos os membros do Set
# Note: a ordem pode variar pois Sets são não ordenados
membros = client.smembers(chave_set)
print(f"🏷️ Tags do Post (repare que 'nosql' aparece apenas uma vez): {membros}")

# SISMEMBER verifica se um valor pertence ao Set (retorna True/False)
tem_python = client.sismember(chave_set, 'python')
tem_nosql = client.sismember(chave_set, 'nosql')
print(f"❓ Contém a tag 'python'? {'Sim' if tem_python else 'Não'}")
print(f"❓ Contém a tag 'nosql'? {'Sim' if tem_nosql else 'Não'}")



In [ ]:
# SREM remove um membro específico do Set
client.srem(chave_set, 'tecnologia')
membros_finais = client.smembers(chave_set)
print(f"🗑️ Tags restantes após remover 'tecnologia': {membros_finais}")

# Limpar chave
client.delete(chave_set)

---
## 6. Encerrando a Conexão
É uma boa prática fechar a conexão com o Redis ao final do uso para liberar recursos.

In [ ]:
# Fechar a conexão com o Redis
client.close()
print("🔌 Conexão com Redis encerrada com sucesso.")

---
## 🏁 Conclusão
Parabéns! Você concluiu a introdução ao Redis. Neste notebook, você aprendeu a:
- ✅ Conectar ao Redis em Python utilizando a biblioteca `redis`.
- ✅ Salvar e obter strings com controle de expiração (TTL).
- ✅ Manipular dados estruturados em **Dicionários** (Hashes), **Filas/Pilhas** (Lists) e **Conjuntos sem duplicidade** (Sets).

### 🚀 Próximos Passos
Para continuar se aprofundando no Redis, experimente:
1. **Sorted Sets (ZADD):** Conjuntos ordenados por pontuação — ideais para leaderboards e rankings.
2. **Pipelines (`client.pipeline()`):** Agrupar múltiplos comandos em uma única requisição para melhorar performance.
3. **Pub/Sub (`client.pubsub()`):** Implementar um sistema de publicação e assinatura de mensagens em tempo real.
4. **Transações (`client.pipeline(transaction=True)`):** Garantir atomicidade em operações que envolvem múltiplas chaves.

### 📚 Referências Úteis
- [Documentação oficial do Redis](https://redis.io/docs/)
- [Referência de comandos](https://redis.io/commands)
- [redis-py (biblioteca Python)](https://redis-py.readthedocs.io/)